In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None)

# Load (Kaggle auto-paths; or use /kaggle/input/your-upload/train.csv if uploaded)
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Prepare (drop Id, target = SalePrice)
X_full = train_data.drop(['Id', 'SalePrice'], axis=1)
y = train_data['SalePrice']
X_test_full = test_data.drop(['Id'], axis=1)

# Split for importances (80/20)
X_train_full, X_val_full, y_train, y_val = train_test_split(X_full, y, random_state=1)

# Num/cat cols
numerical_cols = X_full.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_full.select_dtypes(include=['object']).columns.tolist()

# Preprocessor (impute num=0, one-hot cat)
numerical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value=0))])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer(transformers=[('num', numerical_transformer, numerical_cols), ('cat', categorical_transformer, categorical_cols)])

# Full pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=200, random_state=1))])

print(f"Train shape: {X_full.shape}, Num cols: {len(numerical_cols)}, Cat cols: {len(categorical_cols)}")

In [ ]:
# Baseline CV (5-fold on full train)
cv_scores_full = -1 * cross_val_score(pipeline, X_full, y, cv=5, scoring='neg_mean_absolute_error')
mae_full = cv_scores_full.mean()
print(f"Baseline MAE (all features): {mae_full:.2f}")
print(f"Folds: {cv_scores_full.round(2)}")

# Fit for importances
pipeline.fit(X_train_full, y_train)

# Feature names after one-hot
feature_names = (pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols).tolist() + numerical_cols)

# Importances
importances = pipeline.named_steps['model'].feature_importances_
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)

print("\nTop 20 Features:")
print(feature_importance_df.head(20))

In [ ]:
# Results
results = {'N_features': [], 'MAE': []}

for n in [5, 10, 15, 20]:
    top_features = feature_importance_df['feature'].head(n).tolist()
    
    # Extract original cols from one-hot names (split '_' to get base col, e.g., 'MSZoning_RL' -> 'MSZoning')
    original_cols = list(set([f.split('_')[0] for f in top_features if '_' in f]))
    
    # Filter data with original cols
    X_full_top = X_full[original_cols]
    
    # Top num/cat from original
    top_numerical_cols = [col for col in original_cols if col in numerical_cols]
    top_categorical_cols = [col for col in original_cols if col in categorical_cols]
    
    # Rebuild preprocessor (force dense output to avoid sparse error)
    numerical_transformer_top = Pipeline(steps=[('imputer', SimpleImputer(strategy='constant', fill_value=0))]) if top_numerical_cols else 'passthrough'
    categorical_transformer_top = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]) if top_categorical_cols else 'passthrough'
    preprocessor_top = ColumnTransformer(transformers=[('num', numerical_transformer_top, top_numerical_cols), ('cat', categorical_transformer_top, top_categorical_cols)], sparse_threshold=0)
    
    # Top pipeline
    pipeline_top = Pipeline(steps=[('preprocessor', preprocessor_top), ('model', RandomForestRegressor(n_estimators=200, random_state=1))])
    
    # 5-fold CV
    cv_scores_top = -1 * cross_val_score(pipeline_top, X_full_top, y, cv=5, scoring='neg_mean_absolute_error')
    mae_top = cv_scores_top.mean()
    
    # Record
    results['N_features'].append(n)
    results['MAE'].append(mae_top)
    
    print(f"Top {n} Features - MAE: {mae_top:.2f}")
    print(f"  Folds: {cv_scores_top.round(2)}")
    print(f"  Original cols used: {original_cols}")

# Table
results_df = pd.DataFrame(results)
print("\n--- EVALUATION SCORES ---")
print(results_df.round(2))

# Plot (fixed y-axis)
plt.figure(figsize=(8, 5))
plt.plot(results_df['N_features'], results_df['MAE'], marker='o', linewidth=2)
plt.xlabel('Number of Top Features')
plt.ylabel('MAE (Lower = Better)')
plt.title('Housing Performance with Top Features')
plt.grid(True)
plt.axhline(y=mae_full, color='r', linestyle='--', label=f'Baseline: {mae_full:.2f}')
plt.legend()
plt.show()